### Objective

This notebook evaluates the analytical capability of the current data model by analyzing both the `gold.fact_sales` table and the existing marts.

The goal is to:

- understand available metrics and dimensions  
- validate if current marts support key business questions  
- identify gaps, redundancies, and improvement opportunities  
- define how data should be structured to enable efficient, scalable, and cost-effective queries in the analytics processing layer  

This exploratory analysis will guide the design of optimized data models for analytical consumption.

In [0]:
# ========================================
# EXPLORATORY NOTEBOOK
# DATASET: analytics_sales_overview
# ========================================


# ========================================
# 0. CONFIGURATION
# ========================================

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "gold"

DOMAIN = "analytics"
DATASET = "analytics_sales_overview"

FACT_SALES_DATASET = "fact_sales"
MART_SALES_PERFORMANCE_DATASET = "mart_sales_performance"
MART_CUSTOMER_SALES_DATASET = "mart_customer_sales"
MART_CUSTOMER_PRODUCT_MIX_DATASET = "mart_customer_product_mix"

FACT_SALES_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{FACT_SALES_DATASET}"
MART_SALES_PERFORMANCE_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{MART_SALES_PERFORMANCE_DATASET}"
MART_CUSTOMER_SALES_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{MART_CUSTOMER_SALES_DATASET}"
MART_CUSTOMER_PRODUCT_MIX_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{MART_CUSTOMER_PRODUCT_MIX_DATASET}"

In [0]:
# ========================================
# 1. CONTEXT SETUP
# ========================================

print("=" * 80)
print("STARTING ANALYTICS EXPLORATORY: analytics_sales_overview")
print("=" * 80)

# Set catalog
spark.sql(f"USE CATALOG {CATALOG}")

# Set source schema
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")

print("[INFO] Context setup completed successfully.")

In [0]:
# ========================================
# 2. CONFIGURATION SUMMARY
# ========================================

print("=" * 80)
print("ANALYTICS EXPLORATORY NOTEBOOK CONFIGURATION")
print("=" * 80)
print(f"Catalog:                         {CATALOG}")
print(f"Source schema:                   {SOURCE_SCHEMA}")
print(f"Domain:                          {DOMAIN}")
print(f"Dataset:                         {DATASET}")
print(f"Fact sales table:                {FACT_SALES_TABLE}")
print(f"Mart sales performance table:    {MART_SALES_PERFORMANCE_TABLE}")
print(f"Mart customer sales table:       {MART_CUSTOMER_SALES_TABLE}")
print(f"Mart customer product mix table: {MART_CUSTOMER_PRODUCT_MIX_TABLE}")
print("=" * 80)

In [0]:
# ========================================
# 3. PRE-CHECKS
# ========================================

print("[INFO] Checking source table availability...")

spark.sql(f"DESCRIBE TABLE {FACT_SALES_TABLE}")
spark.sql(f"DESCRIBE TABLE {MART_SALES_PERFORMANCE_TABLE}")
spark.sql(f"DESCRIBE TABLE {MART_CUSTOMER_SALES_TABLE}")
spark.sql(f"DESCRIBE TABLE {MART_CUSTOMER_PRODUCT_MIX_TABLE}")

print("[INFO] Pre-checks completed successfully.")

In [0]:
# ========================================
# 4. READ SOURCE DATA
# ========================================

print("[INFO] Loading analytics source tables from Gold layer...")

df_fact_sales = spark.table(FACT_SALES_TABLE)
df_mart_sales_performance = spark.table(MART_SALES_PERFORMANCE_TABLE)
df_mart_customer_sales = spark.table(MART_CUSTOMER_SALES_TABLE)
df_mart_customer_product_mix = spark.table(MART_CUSTOMER_PRODUCT_MIX_TABLE)

print("[INFO] Source tables loaded successfully.")

print("=" * 80)
print("SOURCE DATA ROW COUNTS")
print("=" * 80)
print(f"Fact Sales:                 {df_fact_sales.count():,}")
print(f"Mart Sales Performance:     {df_mart_sales_performance.count():,}")
print(f"Mart Customer Sales:        {df_mart_customer_sales.count():,}")
print(f"Mart Customer Product Mix:  {df_mart_customer_product_mix.count():,}")
print("=" * 80)

# ========================================
# DATASET PREVIEW
# ========================================

print("=" * 80)
print("DATASET PREVIEW — ANALYTICS SOURCES")
print("=" * 80)

print("\n[INFO] Preview: FACT SALES")
print("-" * 80)
display(df_fact_sales.limit(10))

print("\n[INFO] Preview: MART SALES PERFORMANCE")
print("-" * 80)
display(df_mart_sales_performance.limit(10))

print("\n[INFO] Preview: MART CUSTOMER SALES")
print("-" * 80)
display(df_mart_customer_sales.limit(10))

print("\n[INFO] Preview: MART CUSTOMER PRODUCT MIX")
print("-" * 80)
display(df_mart_customer_product_mix.limit(10))

print("[INFO] Preview completed.")

In [0]:
# ========================================
# 5. INITIAL DATA OVERVIEW
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("INITIAL DATA OVERVIEW — ANALYTICS SOURCES")
print("=" * 80)

datasets = {
    "Fact Sales": df_fact_sales,
    "Mart Sales Performance": df_mart_sales_performance,
    "Mart Customer Sales": df_mart_customer_sales,
    "Mart Customer Product Mix": df_mart_customer_product_mix
}

for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(f"DATASET: {name}")
    print("=" * 80)
    
    # Row count
    row_count = df.count()
    print(f"Row Count: {row_count:,}")
    
    # Column count
    column_count = len(df.columns)
    print(f"Column Count: {column_count}")
    
    # Schema
    print("\nSchema:")
    df.printSchema()
    
    # Sample
    print("\nSample Data:")
    display(df.limit(5))
    
    # Null analysis
    print("\nNull Value Analysis:")
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])
    display(null_counts)

print("\n[INFO] Initial data overview completed.")

### Business Questions

This analysis aims to evaluate whether the current data model (fact + marts) can answer key business questions such as:

- What is the total revenue over time?
- How many orders were placed?
- What is the average ticket value?
- Who are the top customers by revenue?
- Which products generate the most revenue?
- What is the sales performance by channel?
- How does customer-product mix behave?

These questions will be used to validate the current marts and identify gaps for future analytics datasets.

### Current Analytical Coverage Assessment

The current data model provides a strong analytical foundation through a well-structured fact table and a set of specialized marts.

From the analysis performed:

- The `fact_sales` table offers a comprehensive and enriched dataset, containing both metrics and business dimensions, enabling flexible and detailed analysis.

- The existing marts provide focused analytical views:
  - `mart_sales_performance` supports product and channel performance analysis over time  
  - `mart_customer_sales` provides a simplified view of customer-level metrics  
  - `mart_customer_product_mix` enables analysis of customer-product relationships with enriched customer attributes  

Overall, the current structure allows answering several key business questions effectively.

At the same time, the analysis suggests opportunities to further enhance the analytical layer by:

- improving consistency across dimensions (e.g., time, channel, customer attributes)  
- enabling more integrated views across business domains  
- simplifying access to commonly used analytical combinations  

These observations will guide the design of future analytics datasets in the processing layer.

In [0]:
# ========================================
# 6. BUSINESS QUESTION — TOTAL REVENUE
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("BUSINESS QUESTION: TOTAL REVENUE")
print("=" * 80)

# FACT
fact_revenue = df_fact_sales.select(
    F.sum("net_sales_amount").alias("total_revenue")
).collect()[0]["total_revenue"]

print(f"[FACT] Total Revenue: {fact_revenue:,.2f}")


# MART SALES PERFORMANCE
msp_revenue = df_mart_sales_performance.select(
    F.sum("net_sales_amount").alias("total_revenue")
).collect()[0]["total_revenue"]

print(f"[MART SALES PERFORMANCE] Total Revenue: {msp_revenue:,.2f}")


# MART CUSTOMER SALES
mcs_revenue = df_mart_customer_sales.select(
    F.sum("net_sales_amount").alias("total_revenue")
).collect()[0]["total_revenue"]

print(f"[MART CUSTOMER SALES] Total Revenue: {mcs_revenue:,.2f}")


# MART CUSTOMER PRODUCT MIX
mcpm_revenue = df_mart_customer_product_mix.select(
    F.sum("net_sales_amount").alias("total_revenue")
).collect()[0]["total_revenue"]

print(f"[MART CUSTOMER PRODUCT MIX] Total Revenue: {mcpm_revenue:,.2f}")

print("\n[INFO] Revenue comparison completed.")

### Revenue KPI Assessment

The Total Revenue metric is consistently calculated across the `fact_sales` table and all existing marts, confirming that the current data model preserves metric integrity across layers.

#### Consistency

- All datasets answer this question accurately  
- Existing marts already provide aggregated structures that simplify access to this KPI  

#### Efficiency

- Calculating this metric directly from `fact_sales` requires scanning a larger and more granular dataset  
- Existing marts are more efficient for high-frequency use cases because they are already partially aggregated  

#### Conclusion

- No additional dataset is required to support this KPI from a functional standpoint  
- For dashboards and recurring analysis, marts should be preferred over direct fact access  
- Future analytics datasets may centralize frequently used KPIs when justified by recurring consumption needs  

In [0]:
# ========================================
# 7. BUSINESS QUESTION — TOTAL ORDERS
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("BUSINESS QUESTION: TOTAL ORDERS")
print("=" * 80)

# FACT
fact_orders = df_fact_sales.select(
    F.countDistinct("pedido_id").alias("total_orders")
).collect()[0]["total_orders"]

print(f"[FACT] Total Orders: {fact_orders:,}")


# MART SALES PERFORMANCE
msp_orders = df_mart_sales_performance.select(
    F.sum("order_count").alias("total_orders")
).collect()[0]["total_orders"]

print(f"[MART SALES PERFORMANCE] Total Orders: {int(msp_orders):,}")


# MART CUSTOMER SALES
mcs_orders = df_mart_customer_sales.select(
    F.sum("order_count").alias("total_orders")
).collect()[0]["total_orders"]

print(f"[MART CUSTOMER SALES] Total Orders: {int(mcs_orders):,}")


# MART CUSTOMER PRODUCT MIX
mcpm_orders = df_mart_customer_product_mix.select(
    F.sum("order_count").alias("total_orders")
).collect()[0]["total_orders"]

print(f"[MART CUSTOMER PRODUCT MIX] Total Orders: {int(mcpm_orders):,}")

print("\n[INFO] Order comparison completed.")

### Orders KPI Assessment

The Total Orders metric reveals the impact of dataset granularity on KPI calculation.

#### Consistency

- The `fact_sales` table and `mart_customer_sales` return the same total number of orders  
- `mart_sales_performance` and `mart_customer_product_mix` return higher values due to duplicated orders across multiple rows  

#### Analysis

- Not all datasets are suitable for this KPI  
- The correct calculation depends on preserving order-level uniqueness  

#### Modeling

- The current structure is valid but requires clear usage guidelines  
- Dataset selection must always consider the underlying grain  

#### Conclusion

- The correct source for Total Orders depends on the analytical context  
- For unique order counts, datasets aggregated at order or customer level should be used  
- Analytics datasets may help standardize KPI definitions and avoid ambiguity  

In [0]:
# ========================================
# 8. BUSINESS QUESTION — AVERAGE TICKET
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("BUSINESS QUESTION: AVERAGE TICKET")
print("=" * 80)

# FACT
fact_avg_ticket = df_fact_sales.select(
    (F.sum("net_sales_amount") / F.countDistinct("pedido_id")).alias("avg_ticket")
).collect()[0]["avg_ticket"]

print(f"[FACT] Average Ticket: {fact_avg_ticket:,.2f}")


# MART SALES PERFORMANCE
msp_avg_ticket = df_mart_sales_performance.select(
    (F.sum("net_sales_amount") / F.sum("order_count")).alias("avg_ticket")
).collect()[0]["avg_ticket"]

print(f"[MART SALES PERFORMANCE] Average Ticket: {msp_avg_ticket:,.2f}")


# MART CUSTOMER SALES
mcs_avg_ticket = df_mart_customer_sales.select(
    (F.sum("net_sales_amount") / F.sum("order_count")).alias("avg_ticket")
).collect()[0]["avg_ticket"]

print(f"[MART CUSTOMER SALES] Average Ticket: {mcs_avg_ticket:,.2f}")


# MART CUSTOMER PRODUCT MIX
mcpm_avg_ticket = df_mart_customer_product_mix.select(
    (F.sum("net_sales_amount") / F.sum("order_count")).alias("avg_ticket")
).collect()[0]["avg_ticket"]

print(f"[MART CUSTOMER PRODUCT MIX] Average Ticket: {mcpm_avg_ticket:,.2f}")

print("\n[INFO] Average ticket comparison completed.")

### Average Ticket Assessment

The Average Ticket metric highlights the impact of dataset granularity on KPI calculation.

#### Consistency

- The `fact_sales` table and `mart_customer_sales` return the same value, confirming correct calculation based on unique orders  
- `mart_sales_performance` and `mart_customer_product_mix` return lower values due to order duplication across multiple rows  

#### Analysis

- Average Ticket is sensitive to dataset grain  
- Correct calculation requires preserving order-level uniqueness  

#### Modeling

- Not all marts are suitable for this KPI  
- KPI definitions must be aligned with dataset granularity  

#### Conclusion

- Average Ticket should be calculated using datasets aggregated at order or customer level  
- Analytics datasets may standardize KPI definitions and ensure consistent results across use cases  

In [0]:
# ========================================
# 9. BUSINESS QUESTION — TOP CUSTOMERS
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("BUSINESS QUESTION: TOP CUSTOMERS (BY REVENUE)")
print("=" * 80)

# FACT
print("\n[FACT] Top 5 Customers:")
fact_top_customers = (
    df_fact_sales
    .groupBy("cliente_id")
    .agg(F.sum("net_sales_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(5)
    .collect()
)

for row in fact_top_customers:
    print(f"Customer: {row['cliente_id']} | Revenue: {row['total_revenue']:,.2f}")


# MART CUSTOMER SALES
print("\n[MART CUSTOMER SALES] Top 5 Customers:")
mcs_top_customers = (
    df_mart_customer_sales
    .groupBy("cliente_id")
    .agg(F.sum("net_sales_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(5)
    .collect()
)

for row in mcs_top_customers:
    print(f"Customer: {row['cliente_id']} | Revenue: {row['total_revenue']:,.2f}")


# MART CUSTOMER PRODUCT MIX
print("\n[MART CUSTOMER PRODUCT MIX] Top 5 Customers:")
mcpm_top_customers = (
    df_mart_customer_product_mix
    .groupBy("cliente_id")
    .agg(F.sum("net_sales_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(5)
    .collect()
)

for row in mcpm_top_customers:
    print(f"Customer: {row['cliente_id']} | Revenue: {row['total_revenue']:,.2f}")

print("\n[INFO] Top customers comparison completed.")

### Top Customers Assessment

The Top Customers analysis shows consistent results across all evaluated datasets.

#### Consistency

- The `fact_sales`, `mart_customer_sales`, and `mart_customer_product_mix` datasets return identical rankings and revenue values  

#### Analysis

- All datasets are capable of supporting this type of analysis  
- The choice of dataset should be driven by efficiency and required level of detail  

#### Modeling

- `mart_customer_sales` is the most appropriate dataset for this use case, as it is already aggregated at the customer level  
- `mart_customer_product_mix` includes additional product-level granularity that may not be necessary  
- The `fact_sales` table provides full flexibility but requires higher computational cost  

#### Conclusion

- Customer-level marts are well-designed and effectively support customer analysis  
- For most use cases, `mart_customer_sales` is the most efficient and appropriate dataset  
- No additional analytics dataset is required for this type of analysis  

In [0]:
# ========================================
# 10. BUSINESS QUESTION — TOP PRODUCTS
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("BUSINESS QUESTION: TOP PRODUCTS (BY REVENUE)")
print("=" * 80)

# FACT
print("\n[FACT] Top 5 Products:")
fact_top_products = (
    df_fact_sales
    .groupBy("produto_id", "produto_nome")
    .agg(F.sum("net_sales_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(5)
    .collect()
)

for row in fact_top_products:
    print(f"Product: {row['produto_id']} | {row['produto_nome']} | Revenue: {row['total_revenue']:,.2f}")


# MART SALES PERFORMANCE
print("\n[MART SALES PERFORMANCE] Top 5 Products:")
msp_top_products = (
    df_mart_sales_performance
    .groupBy("produto_id", "produto_nome")
    .agg(F.sum("net_sales_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(5)
    .collect()
)

for row in msp_top_products:
    print(f"Product: {row['produto_id']} | {row['produto_nome']} | Revenue: {row['total_revenue']:,.2f}")


# MART CUSTOMER PRODUCT MIX
print("\n[MART CUSTOMER PRODUCT MIX] Top 5 Products:")
mcpm_top_products = (
    df_mart_customer_product_mix
    .groupBy("produto_id", "produto_nome")
    .agg(F.sum("net_sales_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(5)
    .collect()
)

for row in mcpm_top_products:
    print(f"Product: {row['produto_id']} | {row['produto_nome']} | Revenue: {row['total_revenue']:,.2f}")

print("\n[INFO] Top products comparison completed.")

### Top Products Assessment

The Top Products analysis shows consistent results across all evaluated datasets.

#### Consistency

- The `fact_sales`, `mart_sales_performance`, and `mart_customer_product_mix` datasets return identical rankings and revenue values  

#### Analysis

- All datasets support product-based analysis  
- The choice should be based on efficiency and the required level of detail  

#### Modeling

- `mart_sales_performance` is the most suitable dataset, as it is designed for product and channel performance analysis  
- `mart_customer_product_mix` includes additional customer-level granularity that may not be necessary  
- The `fact_sales` table provides full flexibility but requires higher computational cost  

#### Conclusion

- Product-level marts are well-designed and effectively support product performance analysis  
- For most use cases, `mart_sales_performance` is the most efficient and appropriate dataset  
- No additional analytics dataset is required for this type of analysis  

In [0]:
# ========================================
# 11. BUSINESS QUESTION — CUSTOMER + PRODUCT + CHANNEL
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("BUSINESS QUESTION: CUSTOMER + PRODUCT + CHANNEL")
print("=" * 80)

# FACT
print("\n[FACT] Top 5 combinations (Customer + Product + Channel):")
fact_combo = (
    df_fact_sales
    .groupBy("cliente_id", "produto_id", "nome_canal")
    .agg(F.sum("net_sales_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(5)
    .collect()
)

for row in fact_combo:
    print(f"Customer: {row['cliente_id']} | Product: {row['produto_id']} | Channel: {row['nome_canal']} | Revenue: {row['total_revenue']:,.2f}")


# MART SALES PERFORMANCE (no customer)
print("\n[MART SALES PERFORMANCE] Not applicable (no customer dimension)")


# MART CUSTOMER SALES (no product)
print("\n[MART CUSTOMER SALES] Not applicable (no product dimension)")


# MART CUSTOMER PRODUCT MIX (no channel)
print("\n[MART CUSTOMER PRODUCT MIX] Partial (no channel dimension):")

mcpm_combo = (
    df_mart_customer_product_mix
    .groupBy("cliente_id", "produto_id")
    .agg(F.sum("net_sales_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(5)
    .collect()
)

for row in mcpm_combo:
    print(f"Customer: {row['cliente_id']} | Product: {row['produto_id']} | Revenue: {row['total_revenue']:,.2f}")

print("\n[INFO] Combination analysis completed.")

### Multi-Dimensional Analysis Assessment

The Customer + Product + Channel analysis reveals a gap in the current data model.

#### Coverage

- Only the `fact_sales` table supports this analysis, as it contains all required dimensions  
- None of the existing marts provide a complete view of this combination  

#### Analysis

- Multi-dimensional analysis across customer, product, and channel is not fully supported by the current marts  
- This type of query requires either direct access to the fact table or a new aggregated dataset  

#### Modeling

- `mart_sales_performance` lacks customer-level information  
- `mart_customer_sales` lacks product-level information  
- `mart_customer_product_mix` lacks channel-level information  

#### Performance

- Queries on `fact_sales` can be computationally expensive for frequent analytical use  

#### Conclusion

- The current data model does not fully support multi-dimensional analytical queries across all business domains  
- A dedicated analytics dataset combining customer, product, and channel dimensions improves usability and performance  
- This represents a strong candidate for implementation in the analytics processing layer  

### Proposed Analytics Dataset Design

Based on the previous analysis, a gap was identified in supporting multi-dimensional queries combining customer, product, and channel.

To address this, a new analytics dataset is proposed with the following characteristics:

#### Objective
Enable efficient and consistent analysis across customer, product, and channel dimensions, supporting common business questions without requiring complex queries on the fact table.

#### Grain
One row per:
- `data_pedido`
- `cliente_id`
- `produto_id`
- `canal_id`

#### Dimensions
- data_pedido  
- calendar attributes (year, month, etc.)  
- cliente_id  
- produto_id  
- canal_id  

#### Metrics
- quantity_sold  
- net_sales_amount  
- gross_sales_amount  
- total_cost_amount  
- gross_margin_amount  
- order_count  
- line_count  

#### Expected Benefits
- Simplifies multi-dimensional analysis  
- Reduces dependency on the fact table  
- Improves query performance for frequent analytical use cases  
- Provides a consistent and reusable structure for BI and reporting  

This dataset is a strong candidate for implementation in the analytics processing layer.

In [0]:
# ========================================
# 12. CANDIDATE ANALYTICS DATASET PROTOTYPE
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("CANDIDATE ANALYTICS DATASET PROTOTYPE")
print("=" * 80)

df_candidate_analytics_sales = (
    df_fact_sales
    .groupBy(
        "data_pedido",
        "calendar_year",
        "calendar_quarter",
        "calendar_month",
        "calendar_month_name",
        "cliente_id",
        "tipo_cliente",
        "cliente_cidade",
        "distrito",
        "segmento",
        "potencial_valor",
        "cluster_comercial",
        "status_cliente",
        "produto_id",
        "produto_nome",
        "categoria",
        "marca",
        "status_produto",
        "canal_id",
        "nome_canal",
        "descricao_canal"
    )
    .agg(
        F.sum("quantity_sold").alias("quantity_sold"),
        F.sum("gross_sales_amount").alias("gross_sales_amount"),
        F.sum("net_sales_amount").alias("net_sales_amount"),
        F.sum("total_cost_amount").alias("total_cost_amount"),
        F.sum("gross_margin_amount").alias("gross_margin_amount"),
        F.countDistinct("pedido_id").alias("order_count"),
        F.sum("line_count").alias("line_count")
    )
)

print("[INFO] Candidate analytics dataset created in memory.")

print("=" * 80)
print("CANDIDATE DATASET ROW COUNT")
print("=" * 80)
print(f"Rows: {df_candidate_analytics_sales.count():,}")
print("=" * 80)

print("\n[INFO] Candidate dataset preview:")
display(df_candidate_analytics_sales.limit(10))

In [0]:
# ========================================
# 13. GRAIN VALIDATION
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("GRAIN VALIDATION")
print("=" * 80)

grain_columns = [
    "data_pedido",
    "cliente_id",
    "produto_id",
    "canal_id"
]

total_rows = df_candidate_analytics_sales.count()

distinct_grain_rows = (
    df_candidate_analytics_sales
    .select(grain_columns)
    .distinct()
    .count()
)

duplicate_rows = total_rows - distinct_grain_rows

print(f"Total Rows:            {total_rows:,}")
print(f"Distinct Grain Rows:   {distinct_grain_rows:,}")
print(f"Duplicate Grain Rows:  {duplicate_rows:,}")

if duplicate_rows == 0:
    print("[INFO] Grain validation passed. No duplicate grain records found.")
else:
    print("[WARNING] Grain validation failed. Duplicate grain records found.")

In [0]:
# ========================================
# 14. NULL CHECK
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("NULL CHECK")
print("=" * 80)

critical_columns = [
    "data_pedido",
    "cliente_id",
    "produto_id",
    "canal_id",
    "net_sales_amount"
]

null_counts = df_candidate_analytics_sales.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in critical_columns
])

null_counts.show(truncate=False)

print("\n[INFO] Null check completed.")

In [0]:
# ========================================
# 15. DUPLICATE CHECK
# ========================================

from pyspark.sql import functions as F

print("=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)

grain_columns = [
    "data_pedido",
    "cliente_id",
    "produto_id",
    "canal_id"
]

duplicates = (
    df_candidate_analytics_sales
    .groupBy(grain_columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicates.count()

print(f"Number of duplicate grain combinations: {duplicate_count}")

if duplicate_count == 0:
    print("[INFO] No duplicate records found for the defined grain.")
else:
    print("[WARNING] Duplicate records detected.")
    display(duplicates.limit(10))

### Analytics Dataset Readiness Assessment

The proposed analytics dataset was successfully validated through structural and data quality checks.

From a grain perspective:

- The dataset is defined at the level of `data_pedido`, `cliente_id`, `produto_id`, and `canal_id`  
- Validation confirms that each combination is unique, ensuring no duplication and consistent aggregation  

From a data quality perspective:

- Critical fields such as date, customer, product, and channel contain no null values  
- Key metrics are fully populated and reliable for analytical use  

From an analytical perspective:

- The dataset supports multi-dimensional analysis across customer, product, and channel  
- It directly addresses gaps identified in the existing marts  

From a performance perspective:

- The dataset reduces reliance on the fact table  
- Aggregation at this level provides a more efficient structure for recurring analytical queries  

Conclusion:

- The dataset is structurally sound and analytically complete  
- It is ready to be implemented in the analytics processing layer  
- It provides a strong foundation for scalable and efficient analytical consumption

### Final Conclusion

The analysis confirms that the current data model provides a solid foundation for analytical use.

Key findings:

- The `fact_sales` table is comprehensive and supports all analytical scenarios, but requires higher computational effort  
- Existing marts are well-designed and effectively support specific use cases such as customer and product analysis  
- However, they do not fully support multi-dimensional queries combining customer, product, and channel  

The exploratory analysis identified a clear opportunity to enhance the analytics layer by introducing a new dataset with a well-defined grain.

The proposed analytics dataset:

- successfully integrates customer, product, and channel dimensions  
- maintains data consistency and quality  
- improves usability and performance for complex analytical queries  

Final decision:

- Existing marts remain valid and should continue to be used for their respective domains  
- A new analytics dataset should be implemented in the analytics processing layer to support more advanced and integrated analysis  

Next step:

- implement the dataset as a processing notebook  
- apply performance optimizations using Delta Lake capabilities (Liquid Clustering and Auto Optimize)  
- make it available for BI and analytical consumption  

### Complementary Analytics Enhancement — Salesperson Analysis Support

After the first Power BI dashboard iteration, an additional analytical requirement emerged: salesperson performance analysis.

The original `analytics_sales_overview` dataset was designed to support analysis by:

- date
- customer
- product
- sales channel

The dashboard exploration phase demonstrated the need to also support:

- salesperson contribution analysis
- salesperson ranking
- commercial performance segmentation

To support this complementary BI requirement, the analytics layer will be minimally enriched with:

- `vendedor_id`

This enhancement intentionally preserves:

- the existing architecture
- the current metrics
- the aggregation logic
- the table optimization strategy
- the orchestration structure
- the BI consumption purpose of the dataset

The objective is to evolve the analytics layer incrementally without introducing unnecessary structural refactoring.

In [0]:
# Databricks notebook source

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "gold"

FACT_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.fact_sales"

fact_df = spark.table(FACT_TABLE)

display(
    fact_df
    .select("vendedor_id")
    .dropDuplicates()
    .orderBy("vendedor_id")
)

### Salesperson Attribute Validation

The Gold fact table already contains the `vendedor_id` attribute.

This allows the analytics dataset to support salesperson analysis without introducing additional joins or dependencies at this stage.

Current validation confirms:

- `vendedor_id` is available directly in `fact_sales`
- no additional enrichment source is required
- the enhancement can be implemented with a minimal processing change

At this phase, salesperson descriptive attributes such as salesperson name are intentionally excluded to preserve minimal implementation scope.

In [0]:
from pyspark.sql import functions as F

current_grain_columns = [
    "data_pedido",
    "cliente_id",
    "produto_id",
    "canal_id"
]

candidate_grain_columns = current_grain_columns + ["vendedor_id"]

current_grain_rows = (
    fact_df
    .select(current_grain_columns)
    .dropDuplicates()
    .count()
)

candidate_grain_rows = (
    fact_df
    .select(candidate_grain_columns)
    .dropDuplicates()
    .count()
)

grain_difference = candidate_grain_rows - current_grain_rows

print(f"Current grain rows:   {current_grain_rows:,}")
print(f"Candidate grain rows: {candidate_grain_rows:,}")
print(f"Additional grain rows after vendedor_id inclusion: {grain_difference:,}")

### Analytical Grain Evolution

The validation confirms that adding `vendedor_id` introduces an additional analytical dimension to the dataset and therefore evolves the analytical grain of `analytics_sales_overview`.

Original analytical grain:

- `data_pedido`
- `cliente_id`
- `produto_id`
- `canal_id`

Enhanced analytical grain:

- `data_pedido`
- `cliente_id`
- `produto_id`
- `canal_id`
- `vendedor_id`

Validation results:

- Original grain rows: 202,623
- Enhanced grain rows: 209,842
- Additional analytical combinations introduced: 7,219

This result confirms that salesperson is not merely a descriptive attribute, but a legitimate analytical dimension capable of generating additional business segmentation.

The enhancement is considered valid because:

- the existing business metrics remain semantically correct
- aggregation logic remains unchanged
- the dataset purpose remains analytical consumption
- the additional granularity is aligned with BI exploration requirements identified during dashboard development

This evolution was intentionally designed as a minimal-change enhancement.

No changes are required for:

- metric calculation logic
- optimization strategy
- orchestration structure
- storage strategy
- Delta optimization approach
- analytics consumption architecture

The processing notebook will therefore be updated minimally by:

- adding `vendedor_id` to the `SELECT`
- adding `vendedor_id` to the `GROUP BY`
- updating the official analytical grain validation definition

In [0]:
from pyspark.sql import functions as F

metrics = [
    "quantity_sold",
    "gross_sales_amount",
    "net_sales_amount",
    "total_cost_amount",
    "gross_margin_amount",
    "line_count"
]

original_totals = (
    fact_df
    .groupBy(
        "data_pedido",
        "cliente_id",
        "produto_id",
        "canal_id"
    )
    .agg(*[F.sum(c).alias(c) for c in metrics])
    .agg(*[F.sum(c).alias(c) for c in metrics])
    .collect()[0]
    .asDict()
)

enhanced_totals = (
    fact_df
    .groupBy(
        "data_pedido",
        "cliente_id",
        "produto_id",
        "canal_id",
        "vendedor_id"
    )
    .agg(*[F.sum(c).alias(c) for c in metrics])
    .agg(*[F.sum(c).alias(c) for c in metrics])
    .collect()[0]
    .asDict()
)

print("=" * 80)
print("METRIC TOTAL VALIDATION")
print("=" * 80)

for metric in metrics:

    original_value = original_totals[metric]
    enhanced_value = enhanced_totals[metric]

    validation_status = (
        "MATCH"
        if original_value == enhanced_value
        else "DIFFERENT"
    )

    print(f"""
Metric:            {metric}
Original Total:    {original_value}
Enhanced Total:   {enhanced_value}
Validation:        {validation_status}
""")

In [0]:
null_vendedor_id = (
    fact_df
    .filter(F.col("vendedor_id").isNull())
    .count()
)

print(f"Null vendedor_id rows: {null_vendedor_id:,}")

%md
### Complementary Validation Conclusion

The complementary validation confirms that the inclusion of `vendedor_id` is analytically safe and semantically valid for the evolution of `analytics_sales_overview`.

Validation results confirmed that:

- all core business metrics remain identical after the grain enhancement
- no aggregation inconsistencies were introduced
- no null values exist in `vendedor_id`
- salesperson behaves as a valid analytical dimension
- the enhancement preserves the intended BI consumption purpose of the dataset

The validation also confirms that the analytical change introduces additional granularity without impacting overall business totals.

This means the enhancement:

- preserves metric integrity
- preserves analytical consistency
- preserves aggregation correctness
- preserves the current architecture strategy

The dataset is therefore considered ready for a minimal-change processing enhancement in the official analytics processing notebook.